# 02 — Table 1: Mechanism summary across the four CPA n=100 conditions

Reproduces Table 1 of the manuscript: a 7-column summary across the four conditions, anchoring central tendency, instability, and mechanism in one place.

**Manuscript reference:** Table 1; Results §3.2, §3.3, §3.4.

**Columns:**
- Cell type, Split — IDs.
- Raw median — central tendency.
- Sign flips — instability quantification (negative-correlation seeds per 100).
- LOO_max median — mechanism magnitude.
- Top driver in high-LOO seeds — recurrent mode driver with count.
- Wins. range Δ — percent change in raw-Pearson range under p5/p95 leverage clipping.

**Sources:**
- `precomputed/eval/stage5_comparison_n100.csv` (medians, sign flips, LOO_max median, top driver mode and count).
- `precomputed/eval/diag_winsorise_n100_summary.csv` (winsorisation range percent change).

**Outputs:**
- `precomputed/tables/table1_mechanism_summary.csv`


In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from perturb_style import PRECOMPUTED_DIR


## Load data

In [2]:
stage5 = pd.read_csv(PRECOMPUTED_DIR / "eval" / "stage5_comparison_n100.csv")
wins = pd.read_csv(PRECOMPUTED_DIR / "eval" / "diag_winsorise_n100_summary.csv")

print("stage5_comparison_n100.csv:")
print(stage5.to_string(index=False))
print(f"\nwinsorise summary rows: {len(wins)}")


stage5_comparison_n100.csv:
      condition   n  median    MAD    IQR  p5_p95_width  sign_flips  LOO_max_median  top1_frac_median  n_LOOmax_gt_0.15  n_high_LOO_seeds top_driver_mode  top_driver_mode_count_among_high_LOO
    K562 random 100  0.0604 0.1742 0.3358        0.7631          42          0.1633            0.5093                56                56           MED12                                    23
K562 stratified 100  0.1006 0.1516 0.3317        0.7688          40          0.1629            0.4899                53                53           MED17                                    21
    RPE1 random 100  0.2898 0.1001 0.2175        0.4732           5          0.0879            0.4285                13                13           SF3B2                                     8
RPE1 stratified 100  0.2629 0.1120 0.2218        0.4831           8          0.0922            0.4224                 9                 9           SF3B2                                     8

winsorise s

## Build Table 1

Merge the two sources on `condition` and format each cell for display.

In [3]:
ORDER = ["K562 random", "K562 stratified", "RPE1 random", "RPE1 stratified"]

merged = stage5.merge(wins[["condition", "range_pct_change"]], on="condition", how="left")
merged["condition"] = pd.Categorical(merged["condition"], categories=ORDER, ordered=True)
merged = merged.sort_values("condition").reset_index(drop=True)

def fmt_sign(x):
    return f"{x:+.3f}"

def fmt_pct(x):
    return f"{x:+.0f}%"

table1 = pd.DataFrame({
    "Cell type":   [c.split()[0] for c in merged["condition"]],
    "Split":       [c.split()[1] for c in merged["condition"]],
    "Raw median":  [fmt_sign(v) for v in merged["median"]],
    "Sign flips":  [f"{int(v)}/100" for v in merged["sign_flips"]],
    "LOO_max median": [f"{v:.3f}" for v in merged["LOO_max_median"]],
    "Top driver in high-LOO seeds": [
        f"{drv} ({int(cnt)}/{int(tot)})"
        for drv, cnt, tot in zip(merged["top_driver_mode"],
                                  merged["top_driver_mode_count_among_high_LOO"],
                                  merged["n_high_LOO_seeds"])
    ],
    "Wins. range Delta": [fmt_pct(v) for v in merged["range_pct_change"]],
})

print(table1.to_string(index=False))


Cell type      Split Raw median Sign flips LOO_max median Top driver in high-LOO seeds Wins. range Delta
     K562     random     +0.060     42/100          0.163                MED12 (23/56)              -22%
     K562 stratified     +0.101     40/100          0.163                MED17 (21/53)              -16%
     RPE1     random     +0.290      5/100          0.088                 SF3B2 (8/13)               +4%
     RPE1 stratified     +0.263      8/100          0.092                  SF3B2 (8/9)               +5%


## Save

In [4]:
out_dir = PRECOMPUTED_DIR / "tables"
out_dir.mkdir(exist_ok=True, parents=True)
out_path = out_dir / "table1_mechanism_summary.csv"
table1.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
table1


Saved: precomputed/tables/table1_mechanism_summary.csv


,Cell type,Split,Raw median,Sign flips,LOO_max median,Top driver in high-LOO seeds,Wins. range Delta
0,K562,random,+0.060,42/100,0.163,MED12 (23/56),-22%
1,K562,stratified,+0.101,40/100,0.163,MED17 (21/53),-16%
2,RPE1,random,+0.290,5/100,0.088,SF3B2 (8/13),+4%
3,RPE1,stratified,+0.263,8/100,0.092,SF3B2 (8/9),+5%
